# NeoOLAF — FINAL 2 FinCausal on OpenRouter + final ESL/FinCausal evaluation

This notebook finishes the existing full experiment.

Only these unresolved FinCausal records are eligible:
- `fincausal:202:487588ceb3b14bb5`
- `fincausal:709:80f02bc8ff7bd772`

Execution:
- **2 document processes concurrently**
- **4 NeoOLAF layer workers per document**
- `openai/gpt-oss-20b` through OpenRouter
- frozen FinCausal config `unified-v1.3.1-selection-hotfix`

EventStoryLine is **never rerun**; its 443/443 result is re-evaluated offline.  
After the final FinCausal runs, the notebook re-evaluates both full datasets and writes a final JSON summary.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, time, subprocess, textwrap, re, shutil, uuid as _uuid
from pprint import pprint

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def safe_dir_name(x):
    return re.sub(r"[^A-Za-z0-9_.-]+","_",str(x))[:120]

def find_project_root():
    candidates=[]
    env=os.environ.get("NEOOLAF_PROJECT_ROOT")
    if env: candidates.append(Path(env))
    cwd=Path.cwd().resolve()
    candidates.extend([cwd,*cwd.parents])
    candidates.append(Path(r"C:\Users\galencarmedeiro\NeoOLAF"))
    for p in candidates:
        if (p/"src"/"neoolaf").is_dir() and (p/"examples").is_dir():
            return p.resolve()
    raise FileNotFoundError("Set NEOOLAF_PROJECT_ROOT.")

PROJECT_ROOT=find_project_root()
EXPERIMENT_ROOT=PROJECT_ROOT/"examples"/"RAGTreeDatasets"
TOOLS_DIR=EXPERIMENT_ROOT/"tools"
for p in [PROJECT_ROOT,PROJECT_ROOT/"src",TOOLS_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0,str(p))

import ragtree_experiment_state_v1 as expstate

RUNS_ROOT=EXPERIMENT_ROOT/"runs"/"full_process_isolated_eventstoryline_fincausal_v1"
PROGRESS_PATH=RUNS_ROOT/"full_progress.json"
USAGE_SESSIONS_PATH=RUNS_ROOT/"openrouter_usage_sessions.json"
FINAL_SUMMARY_PATH=RUNS_ROOT/"FINAL_ESL_FINCAUSAL_RESULTS.json"

MODEL_NAME="openai/gpt-oss-20b"
OPENROUTER_HOST="https://openrouter.ai/api/v1"
DOCUMENT_WORKERS=2
LAYER_WORKERS=4
RUN_PAID_FINAL2=True

EXPECTED_PENDING_KEYS={
    "fincausal:202:487588ceb3b14bb5",
    "fincausal:709:80f02bc8ff7bd772",
}

LOG_ROOT=RUNS_ROOT/"_worker_console_logs"/"fincausal_final2_parallel2x4"
BACKUP_ROOT=RUNS_ROOT/"_failed_run_backups_before_final2"
LOG_ROOT.mkdir(parents=True,exist_ok=True)
BACKUP_ROOT.mkdir(parents=True,exist_ok=True)

assert PROGRESS_PATH.is_file(), PROGRESS_PATH
print("PROJECT_ROOT:",PROJECT_ROOT)
print("Final FinCausal execution: 2 documents concurrently × 4 layer workers each")


PROJECT_ROOT: C:\Users\galencarmedeiro\NeoOLAF
Final FinCausal execution: 2 documents concurrently × 4 layer workers each


## Verify existing state — zero API calls

In [2]:
RAGTREE_ROOT=expstate.discover_ragtree_root(PROJECT_ROOT)
PRE=expstate.discover_preprocessed_dir(RAGTREE_ROOT)
DATA=expstate.locate_dataset_files(PRE)

dataset_rows={
    "eventstoryline":expstate.read_jsonl(DATA["eventstoryline"]),
    "fincausal":expstate.read_jsonl(DATA["fincausal"]),
}
assert len(dataset_rows["eventstoryline"])==443
assert len(dataset_rows["fincausal"])==967

progress=json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
assert progress["model"]==MODEL_NAME
assert progress["host"]==OPENROUTER_HOST
assert progress["datasets"]["eventstoryline"]["version"]=="v1.7"
assert progress["datasets"]["fincausal"]["version"]=="unified-v1.3.1-selection-hotfix"

esl_state=progress["datasets"]["eventstoryline"]
fc_state=progress["datasets"]["fincausal"]

assert len(esl_state.get("completed_record_keys") or [])==443
assert esl_state.get("fully_finished_at_utc")

fc_completed=set(fc_state.get("completed_record_keys") or [])
pending_rows=[
    r for r in dataset_rows["fincausal"]
    if expstate.record_key("fincausal",r) not in fc_completed
]
pending_keys={expstate.record_key("fincausal",r) for r in pending_rows}

assert 965 <= len(fc_completed) <= 967
assert pending_keys.issubset(EXPECTED_PENDING_KEYS), pending_keys

for r in pending_rows:
    clean=expstate.strip_gold(r)
    assert not ({"entities","relations","pred_relations","ontology_links"} & set(clean))

print("EventStoryLine:",len(esl_state["completed_record_keys"]),"/443 COMPLETE")
print("FinCausal:",len(fc_completed),"/967")
print("Pending:")
for r in pending_rows:
    print(" -",expstate.record_key("fincausal",r),"|",r.get("title"))


EventStoryLine: 443 /443 COMPLETE
FinCausal: 965 /967
Pending:
 - fincausal:202:487588ceb3b14bb5 | 0387.00002.2
 - fincausal:709:80f02bc8ff7bd772 | 37


## Offline evaluator

In [3]:
def atomic_json(path,obj):
    expstate.atomic_write_json(Path(path),obj)

def result_path(dataset_key,rkey):
    return RUNS_ROOT/dataset_key/"full"/safe_dir_name(rkey)/"posthoc_evaluation.json"

def mc(m,*names):
    for n in names:
        if isinstance(m,dict) and m.get(n) is not None:
            return int(m.get(n) or 0)
    return 0

def norm(m):
    tp=mc(m,"tp","true_positive")
    fp=mc(m,"fp","false_positive")
    fn=mc(m,"fn","false_negative")
    pred=mc(m,"pred","predicted") or (tp+fp)
    gold=mc(m,"gold","gold_unique") or (tp+fn)
    return {"tp":tp,"fp":fp,"fn":fn,"pred":pred,"gold":gold}

def prf(tp,fp,fn):
    p=tp/(tp+fp) if tp+fp else 0.0
    r=tp/(tp+fn) if tp+fn else 0.0
    f1=2*p*r/(p+r) if p+r else 0.0
    return p,r,f1

def reload_progress():
    global progress
    progress=json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))

def load_results(dataset_key):
    completed=set(progress["datasets"][dataset_key].get("completed_record_keys") or [])
    out=[]
    for row in dataset_rows[dataset_key]:
        rk=expstate.record_key(dataset_key,row)
        if rk not in completed: continue
        p=result_path(dataset_key,rk)
        assert p.is_file(),(rk,p)
        out.append(json.loads(p.read_text(encoding="utf-8")))
    return out

def aggregate(dataset_key):
    results=load_results(dataset_key)
    relc=[norm(r.get("relation_metrics") or {}) for r in results]
    epc=[norm(r.get("endpoint_metrics") or {}) for r in results]

    def agg(cs):
        tp=sum(x["tp"] for x in cs); fp=sum(x["fp"] for x in cs); fn=sum(x["fn"] for x in cs)
        pred=sum(x["pred"] for x in cs); gold=sum(x["gold"] for x in cs)
        p,r,f1=prf(tp,fp,fn)
        return {"pred":pred,"gold":gold,"tp":tp,"fp":fp,"fn":fn,
                "precision":p,"recall":r,"micro_f1":f1}

    rel=agg(relc); ep=agg(epc)
    doc_f1=[float((r.get("relation_metrics") or {}).get("f1",0.0) or 0.0) for r in results]
    pos_f1=[
        float((r.get("relation_metrics") or {}).get("f1",0.0) or 0.0)
        for r,c in zip(results,relc) if c["gold"]>0
    ]
    rel["macro_doc_f1"]=sum(doc_f1)/len(doc_f1) if doc_f1 else 0.0
    rel["macro_positive_gold_doc_f1"]=sum(pos_f1)/len(pos_f1) if pos_f1 else 0.0

    ds=progress["datasets"][dataset_key]
    completed=len(ds.get("completed_record_keys") or [])
    return {
        "dataset":dataset_key,
        "version":ds["version"],
        "completed_records":completed,
        "total_records":ds["total_records"],
        "status":"COMPLETE" if completed==ds["total_records"] else "PARTIAL",
        "relation":rel,
        "endpoint":ep,
    }

def final_summary():
    reload_progress()
    return {
        "experiment":progress["experiment"],
        "model":MODEL_NAME,
        "generated_at_utc":utc_now(),
        "datasets":{
            "eventstoryline":aggregate("eventstoryline"),
            "fincausal":aggregate("fincausal"),
        },
        "scope_note":"Full-scale final evaluation includes EventStoryLine and FinCausal only."
    }

s=final_summary()
for k in ["eventstoryline","fincausal"]:
    a=s["datasets"][k]; r=a["relation"]
    print(k,a["completed_records"],"/",a["total_records"],a["status"],
          "P=",round(r["precision"],6),"R=",round(r["recall"],6),"F1=",round(r["micro_f1"],6))


eventstoryline 443 / 443 COMPLETE P= 0.150509 R= 0.093568 F1= 0.115397
fincausal 965 / 967 PARTIAL P= 0.836957 R= 0.249191 F1= 0.38404


## Build FinCausal final-two OpenRouter worker

In [4]:
WORKER_SCRIPT=RUNS_ROOT/"_final2_fincausal_parallel4_worker.py"

worker_src=r"""
from pathlib import Path
from datetime import datetime, timezone
import argparse, os, sys, json, time, shutil, re

def utc_now(): return datetime.now(timezone.utc).isoformat()
def safe(x): return re.sub(r"[^A-Za-z0-9_.-]+","_",str(x))[:120]

ap=argparse.ArgumentParser()
for n in ["project-root","run-root","record-key","model","host","api-key"]:
    ap.add_argument("--"+n,required=True)
args=ap.parse_args()

PROJECT_ROOT=Path(args.project_root).resolve()
EXP=PROJECT_ROOT/"examples"/"RAGTreeDatasets"
TOOLS=EXP/"tools"
for p in [PROJECT_ROOT,PROJECT_ROOT/"src",TOOLS]:
    if str(p) not in sys.path:
        sys.path.insert(0,str(p))

import ragtree_experiment_state_v1 as expstate
import ragtree_dataset_adapters_v1_8 as adapters

RAG=expstate.discover_ragtree_root(PROJECT_ROOT)
PRE=expstate.discover_preprocessed_dir(RAG)
ONT=expstate.discover_ontology_dir(RAG)
DATA=expstate.locate_dataset_files(PRE)
ONTO=expstate.locate_ontology_files(ONT)

rows=expstate.read_jsonl(DATA["fincausal"])
matches=[r for r in rows if expstate.record_key("fincausal",r)==args.record_key]
if len(matches)!=1:
    raise RuntimeError(f"Expected 1 row, got {len(matches)}")
gold=matches[0]

run_root=Path(args.run_root)
run_dir=run_root/"fincausal"/"full"/safe(args.record_key)
backup_dir=run_root/"_failed_run_backups_before_final2"/safe(args.record_key)

if run_dir.exists() and not backup_dir.exists():
    shutil.copytree(run_dir,backup_dir)
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir(parents=True,exist_ok=True)

cfg={
    "profile":EXP/"configs/fincausal_profile_unified_v1_3.json",
    "guidance":EXP/"configs/fincausal_guidance_unified_v1_3.json",
    "task":EXP/"configs/fincausal_task_guidance_unified_v1_3.json",
    "catalog":EXP/"ontology/fincausal_relation_catalog.json",
    "aliases":EXP/"ontology/fincausal_relation_aliases.json",
}

contract=expstate.gold_contract_summary("fincausal",gold)
clean=expstate.strip_gold(gold)
if {"entities","relations","pred_relations","ontology_links"} & set(clean):
    raise RuntimeError("Gold leakage")

input_path=run_dir/"pipeline_input_NO_GOLD.jsonl"
gold_path=run_dir/"POSTHOC_GOLD_AFTER_LAYER12.jsonl"
expstate.write_jsonl(input_path,[clean])

started=utc_now(); t0=time.perf_counter()

state=adapters.run_native_pipeline_record(
    dataset_key="fincausal",
    project_root=PROJECT_ROOT,
    input_jsonl=input_path,
    ontology_path=ONTO["fincausal"],
    profile_path=cfg["profile"],
    guidance_path=cfg["guidance"],
    task_guidance_path=cfg["task"],
    relation_catalog_path=cfg["catalog"],
    relation_aliases_path=cfg["aliases"],
    run_dir=run_dir,
    model_name=args.model,
    api_key=args.api_key,
    host=args.host,
    workers=4,
    max_tokens=8192,
    request_timeout=180,
    reasoning_effort="minimal",
    verbose=True,
    clean_run_dir=False,
)

expstate.write_jsonl(gold_path,[{k:v for k,v in gold.items() if not k.startswith("__")}])
result=adapters.evaluate_state("fincausal",state,gold)

expected=int(contract["gold_target_relation_count"])
evaluated=int((result.get("relation_metrics") or {}).get("gold",0) or 0)
if expected>0 and evaluated==0:
    raise RuntimeError(f"Evaluator integrity error: expected {expected}, evaluator saw 0")

result.update({
    "dataset":"fincausal",
    "version":"unified-v1.3.1-selection-hotfix",
    "record_key":args.record_key,
    "document_id":gold.get("document_id"),
    "title":gold.get("title"),
    "pre_run_gold_contract":contract,
    "run_dir":str(run_dir),
    "started_at_utc":started,
    "finished_at_utc":utc_now(),
    "elapsed_seconds":time.perf_counter()-t0,
    "model":args.model,
    "host":args.host,
    "execution_provenance":"openrouter_final2_process_parallel_layer_workers_4",
    "layer_workers":4,
    "process_id":os.getpid(),
    "gold_visible_to_pipeline":False,
})
adapters.write_json(run_dir/"posthoc_evaluation.json",result)
print("FINAL2_SUCCESS",args.record_key,json.dumps(result.get("relation_metrics") or {}))
"""

WORKER_SCRIPT.write_text(textwrap.dedent(worker_src),encoding="utf-8")
compile(WORKER_SCRIPT.read_text(encoding="utf-8"),str(WORKER_SCRIPT),"exec")
print("Worker syntax: OK")


Worker syntax: OK


## Run the pending final records — paid cell

In [5]:
def write_usage_sessions():
    atomic_json(
        USAGE_SESSIONS_PATH,
        {
            "experiment":progress["experiment"],
            "model":MODEL_NAME,
            "host":OPENROUTER_HOST,
            "usage_sessions":progress.get("usage_sessions") or [],
        },
    )

def launch(row,api_key):
    rk=expstate.record_key("fincausal",row)
    log_path=LOG_ROOT/f"{safe_dir_name(rk)}.log"
    fh=open(log_path,"w",encoding="utf-8",buffering=1)
    cmd=[
        sys.executable,str(WORKER_SCRIPT),
        "--project-root",str(PROJECT_ROOT),
        "--run-root",str(RUNS_ROOT),
        "--record-key",rk,
        "--model",MODEL_NAME,
        "--host",OPENROUTER_HOST,
        "--api-key",api_key,
    ]
    proc=subprocess.Popen(cmd,stdout=fh,stderr=subprocess.STDOUT,cwd=str(PROJECT_ROOT),env=os.environ.copy())
    return {"rk":rk,"proc":proc,"fh":fh,"log":log_path,"t0":time.monotonic()}

reload_progress()
fc_completed=set(progress["datasets"]["fincausal"].get("completed_record_keys") or [])
pending=[
    r for r in dataset_rows["fincausal"]
    if expstate.record_key("fincausal",r) not in fc_completed
]
assert {expstate.record_key("fincausal",r) for r in pending}.issubset(EXPECTED_PENDING_KEYS)
assert len(pending)<=2

if not pending:
    print("FinCausal already 967/967 — no API calls.")
elif not RUN_PAID_FINAL2:
    print("RUN_PAID_FINAL2=False — no API calls.")
else:
    api_key=os.environ.get("OPENROUTER_API_KEY","").strip().strip('"').strip("'")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is missing.")

    session={
        "session_id":_uuid.uuid4().hex,
        "kind":"fincausal_final2_parallel2x4",
        "started_at_utc":utc_now(),
        "finished_at_utc":None,
        "document_workers":2,
        "layer_workers_per_document":4,
        "pending_at_start":len(pending),
        "completed_at_start":len(progress["datasets"]["fincausal"]["completed_record_keys"]),
        "completed_at_end":None,
    }
    progress.setdefault("usage_sessions",[]).append(session)
    atomic_json(PROGRESS_PATH,progress)
    write_usage_sessions()

    active={}
    failures=[]
    try:
        for row in pending:
            c=launch(row,api_key)
            active[c["rk"]]=c
            print("START",c["proc"].pid,c["rk"])

        while active:
            done=[]
            for rk,info in list(active.items()):
                rc=info["proc"].poll()
                if rc is None:
                    continue
                info["fh"].close()
                done.append(rk)
                if rc==0 and result_path("fincausal",rk).is_file():
                    if rk not in progress["datasets"]["fincausal"]["completed_record_keys"]:
                        progress["datasets"]["fincausal"]["completed_record_keys"].append(rk)
                    progress["datasets"]["fincausal"].setdefault("failures",{}).pop(rk,None)
                    atomic_json(PROGRESS_PATH,progress)
                    print("DONE",rk,f"{time.monotonic()-info['t0']:.1f}s")
                else:
                    failures.append({"record_key":rk,"returncode":rc,"log":str(info["log"])})
                    print("FAILED",rk,"log:",info["log"])
            for rk in done:
                active.pop(rk,None)
            if active:
                time.sleep(1)

        if failures:
            raise RuntimeError(failures)

        if len(progress["datasets"]["fincausal"]["completed_record_keys"])==967:
            progress["datasets"]["fincausal"]["fully_finished_at_utc"]=utc_now()
            progress["datasets"]["fincausal"]["failures"]={}
            atomic_json(PROGRESS_PATH,progress)
            print("FINCAUSAL COMPLETE 967/967")
    finally:
        session["finished_at_utc"]=utc_now()
        session["completed_at_end"]=len(progress["datasets"]["fincausal"]["completed_record_keys"])
        atomic_json(PROGRESS_PATH,progress)
        write_usage_sessions()


START 29468 fincausal:202:487588ceb3b14bb5
START 11760 fincausal:709:80f02bc8ff7bd772
DONE fincausal:202:487588ceb3b14bb5 9.1s
DONE fincausal:709:80f02bc8ff7bd772 42.1s
FINCAUSAL COMPLETE 967/967


## Final EventStoryLine + FinCausal evaluation — zero API calls

In [6]:
s=final_summary()
atomic_json(FINAL_SUMMARY_PATH,s)

print("FINAL FULL RESULTS")
print("==================")
for k in ["eventstoryline","fincausal"]:
    a=s["datasets"][k]
    r=a["relation"]; e=a["endpoint"]
    print(f"\n{k} [{a['version']}] {a['completed_records']}/{a['total_records']} | {a['status']}")
    print(
        f"relation: P={r['precision']:.6f} R={r['recall']:.6f} "
        f"micro-F1={r['micro_f1']:.6f} macro-doc-F1={r['macro_doc_f1']:.6f} "
        f"macro-positive-doc-F1={r['macro_positive_gold_doc_f1']:.6f} "
        f"TP={r['tp']} FP={r['fp']} FN={r['fn']}"
    )
    print(
        f"endpoint: P={e['precision']:.6f} R={e['recall']:.6f} "
        f"micro-F1={e['micro_f1']:.6f} TP={e['tp']} FP={e['fp']} FN={e['fn']}"
    )

print("\nSaved:",FINAL_SUMMARY_PATH)
print("Usage sessions:",USAGE_SESSIONS_PATH)
print("Send me this executed notebook afterward.")


FINAL FULL RESULTS

eventstoryline [v1.7] 443/443 | COMPLETE
relation: P=0.150509 R=0.093568 micro-F1=0.115397 macro-doc-F1=0.111308 macro-positive-doc-F1=0.111308 TP=902 FP=5091 FN=8738
endpoint: P=0.997348 R=0.507126 micro-F1=0.672370 TP=2633 FP=7 FN=2559

fincausal [unified-v1.3.1-selection-hotfix] 967/967 | COMPLETE
relation: P=0.836957 R=0.248654 micro-F1=0.383402 macro-doc-F1=0.238883 macro-positive-doc-F1=0.248654 TP=231 FP=45 FN=698
endpoint: P=1.000000 R=1.000000 micro-F1=1.000000 TP=947 FP=0 FN=0

Saved: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\FINAL_ESL_FINCAUSAL_RESULTS.json
Usage sessions: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\runs\full_process_isolated_eventstoryline_fincausal_v1\openrouter_usage_sessions.json
Send me this executed notebook afterward.
